### This notebook compute "VCA18. Indice del volumen embalsado" indicator for the 27 basins of IKI Project

Spanish: Indice del volumen embalsado

**Created:** 1/9/2026 by Sophia Bakar (sbakar@rti.org)

**Project #:** 0219481  

**Last modified:** 1/12/2026 by Sophia

**Status:** Complete and loaded in SQLite for the baseline and first future scenario. 

**QA Status:** reviewed by  

**Original Script Stored at:** Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\Vulnerabilidad

**Packages:** pandas, numpy, geopandas, sqlite3
 
**Inputs:**   

**Outputs:** 
 
**Assumptions:**

**Future work:** We should think about an approach for 1. When a reservoir falls within more than 1 COMID, should we apply the weighted method based on how much of the reservoir lies within a COMID to get values for more than one COMID? 2. If a COMID has more than one reservoir, do we just consider the total average storage between the 2?
 
**Notes:** 

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import sqlite3
import matplotlib.pyplot as plt
import re 

In [2]:
# set up user and database path
#user = 'jmayo'
#user= 'cpickering'
#user = 'sgilson'
#user = 'nreynolds'
user = 'sbakar'
#db_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db'
db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"
# wateralloc_db = fr"C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"
wateralloc_db = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"

In [3]:
# set up indicator ID and get scenarios from database
IndID= 518 #Indicator ID (Exposure = 2 + 0X where X is the Exposure Indicator number, Peligro= 1 +0x, VSB= 3 +0x, VSS= 4 +0x, VCA= 5 +0x)
conn = sqlite3.connect(db_path)

scenarios_df = pd.read_sql_query(
    """
    SELECT ScnID, ScnName
    FROM ScnMod
    """,
    conn
)

conn.close()

# For now: only baseline and first future
scenario_ids = scenarios_df.loc[
    scenarios_df['ScnID'].isin([1, 2]), 'ScnID'
].tolist()

# for all scenarios:
# scenario_ids = scenarios_df['ScnID'].tolist()


In [4]:
## check what scenarios are available in the WaterALLOC database
# Connect to WaterALLOC database
conn_wa = sqlite3.connect(wateralloc_db)

# Query available scenarios
scenarios_query = """
SELECT DISTINCT Scenario
FROM Scenarios
ORDER BY Scenario
"""

wa_scenarios_df = pd.read_sql_query(scenarios_query, conn_wa)

print("Available scenarios in WaterALLOC DB:")
for s in wa_scenarios_df["Scenario"]:
    print(f" - {s}")



Available scenarios in WaterALLOC DB:
 - CC_CMIP6_85_2050
 - Linea_Base_2020
 - Linea_Base_2020_Embalses


In [5]:
# define filepaths for input data
#subbasins_shapefile = fr"C:/Users/{user}/Research Triangle Institute/IKI Peru Project - General/Interno/AI2b_Modelacion/Grupos_Modelacion/GIS_WaterALLOC_General/Peru_AHD_with_districts.shp"
subbasins_shapefile = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\GIS_WaterALLOC_General\Peru_AHD_with_districts.shp"
# read in input data
subbasins_gdf = gpd.read_file(subbasins_shapefile).to_crs('EPSG:32718')

In [ ]:
# get monthly average flow from WaterALLOC database for each scenario
# (Inflow + Local Flow - Demand)
flow_all = []

conn_wa = sqlite3.connect(wateralloc_db)

for scn_id_dynamic, scenario_name in scenario_mapping.items():
    print(f"\nProcessing scenario: {scenario_name} (ScnID={scn_id_dynamic})")

    query_flow = """
    SELECT 
        a.comid AS COMID,
        AVG(
            a.[Oferta Entrada] 
          + a.[Oferta Local Sup] 
          - a.[Dem Local Sup]
        ) AS Caudal_Medio,
        COUNT(*) AS n_months
    FROM "WAMSS_Balance por COMID (+Indice de estres)" AS a
    JOIN WAMMS_RunsInfo AS b 
        ON a.RunID = b.RunID
    JOIN Scenarios AS c 
        ON c.ScnID = b.ScnID
    WHERE c.Scenario = ?
    GROUP BY a.comid
    """

    flow_df = pd.read_sql_query(query_flow, conn_wa, params=(scenario_name,))

    # Add dynamic ScnID column
    flow_df['ScnID_dynamic'] = scn_id_dynamic

    flow_all.append(flow_df)

conn_wa.close()

flow_all_df = pd.concat(flow_all, ignore_index=True)